In [1]:
import math
import pandas as pd
import numpy as np
import json

# READ DATA (and drop testing workerids)

In [2]:
df = pd.read_csv("./dataRaw/experiment_6323_results.csv")

unnecessary_cols = ["questionid", "filename", "listnumber", "assignmentid", "hitid", "origin", "timestamp", "partid", "id", "imgorder"]
unformatted_df = df.drop(unnecessary_cols, axis=1)

testworkers = ["test", "123", "testFirefox", "testRedirect", "testFinal"]  # add more to this list if there were more test runs
unformatted_df = unformatted_df[~unformatted_df["workerid"].isin(testworkers)]

len(unformatted_df["workerid"].unique())

78

## FORMAT RATING VALUES FROM THE ANSWER COLUMN

In [3]:
def format_answer(answer, weak_word, strong_word, antonym):
    """
    returns multiple results.
    whatever data is needed out of the answer, edit and manipulate it in this function
    """
    results = json.loads(json.loads(answer))
    new_dict = dict()
    
    if not results.get("ratings"):
        return None, None, None, None, None
    for item in results["ratings"]:
        name = item["image"].replace(".png", "")
        new_dict.update(
            {name: item["rating"]}
        )

    if results["speaker"] in ["frank", "bri"]:
        speakerType = "native"
    else:
        speakerType = "nonnative"
    
    if results["speaker"] in ["bri", "sasha"]:
        speakerGender = "female"
    else:
        speakerGender = "male"

    # weak word rating, strong word rating, antonym rating
    return new_dict.get(weak_word), new_dict.get(strong_word), new_dict.get(antonym), speakerType, speakerGender

In [4]:
answers_only_df = unformatted_df.apply(
    lambda x: format_answer(x.answer, x.weak, x.strong, x.antonym), 
    axis=1, 
    result_type="expand",  # this is how you make it into multiple columns
)
answers_only_df.columns=["derivationStrength", "strongRating", "antonymRating", "speakerType", "speakerGender"]
answers_only_df

,derivationStrength,strongRating,antonymRating,speakerType,speakerGender
1,90.0,10.0,0.0,nonnative,male
2,35.0,60.0,5.0,native,female
3,94.0,6.0,0.0,nonnative,male
4,100.0,0.0,0.0,nonnative,male
5,100.0,0.0,0.0,native,female
...,...,...,...,...,...
2712,100.0,0.0,0.0,nonnative,female
2713,100.0,0.0,0.0,native,female
2714,100.0,0.0,0.0,nonnative,female
2715,100.0,0.0,0.0,native,male


## READ WORD FREQ DICT IN

In [5]:
word_freq_dict_df = pd.read_csv("./dataRaw/wordFreqDict.csv")
word_freq_dict_df.head()

word_freq_dict_df[word_freq_dict_df['Frequency'].isin(["x"])]

,Word,Rank,Frequency


## RENAME COLUMN NAMES TO WHAT THE MODEL NEEDS

In [6]:
answers_df = pd.concat([unformatted_df, answers_only_df], axis=1)
answers_df = answers_df.drop(["answer"], axis=1)

answers_df = answers_df.rename(columns={
    "workerid": "participantId",
    "itemid": "itemId",
    "type": "itemType"
})

answers_df[answers_df["participantId"] == "66b153e252bf568e5c9f0ba2"]

,participantId,itemId,itemType,weak,strong,antonym,lowfreq1,lowfreq2,derivationStrength,strongRating,antonymRating,speakerType,speakerGender


# FILTER - EXCLUSION CRITERIA

### Picture Rating Control Trials

In [7]:
# filter out rating scores
#    - if avg. rating across all unambiguous trials were < 80%
umambiguous_trials = answers_df[answers_df["itemType"] == "unambiguous"]
picrating_filtered_participants = list(
    pd.unique(
        umambiguous_trials.groupby("participantId").filter(
            lambda x: x["derivationStrength"].mean() < 80
        )["participantId"]
    )
)
picrating_filtered_participants

[]

### Picture Rating Technical Problems

In [8]:
# a lot of responses for one person didn't get recorded unfortunately
countNaN = umambiguous_trials["derivationStrength"].isnull().groupby(
    umambiguous_trials["participantId"]
).sum().astype(int).reset_index(name="countNaN")

technical_prob_filtered_participants = list(countNaN[countNaN["countNaN"] > 1]["participantId"].unique())
technical_prob_filtered_participants

['69cc1fac2b52b1c4dd21cc7d']

## ToM Filtered Participants

Get a list from the R script:
- technical problems (anything over 10% of responses missing)
- control accuracy < 80%

In [9]:
nathan4u_filtered_participants = [
    # technical issues: 50+ unanswered
    "69c093d33989aa791c4e8b2d", 
    "6a2482e9af72b55296872b0d",
    "6a00dbcc626a780a8e5c07a4",
    "6a232745f21e165e2b54ed58",

    # control accuracy < 80%
    "6965bd850ce22095dd7a84ca",
    "6978d830c5d415696508430d",
    "69a6eb2bc679065ed1cecc51",
    "69ba37faee13f176e7635323",
    "69c093d33989aa791c4e8b2d",
    "69cc1fac2b52b1c4dd21cc7d",
    "69f674db81cb46678aae6391",
    "69fb86296fb6f0919bf31075",
    "6a00dbcc626a780a8e5c07a4",
    "6a14434573bcee3f17b7f762",
    "6a14e67cf20909f60e34ae64",
    "6a232745f21e165e2b54ed58",
    "6a2482e9af72b55296872b0d",
]

## Audio Filtered Participants

After transcribing the audio for the two practice perspective taking trials, any that did not choose the target word will be excluded.

Anyone who did not provide a response that is indicative of understanding the task in the final critical audio explanation will be excluded.

In [10]:
audio_filtered_participants = []

## Total Filtered Participants

In [11]:
all_filtered_participants = list(set(
    picrating_filtered_participants + 
    nathan4u_filtered_participants + 
    audio_filtered_participants + 
    technical_prob_filtered_participants
))

print("total number of filtered participants: ", len(all_filtered_participants))
all_filtered_participants

total number of filtered participants:  13


['69fb86296fb6f0919bf31075',
 '69ba37faee13f176e7635323',
 '6a14e67cf20909f60e34ae64',
 '6a00dbcc626a780a8e5c07a4',
 '6a232745f21e165e2b54ed58',
 '69f674db81cb46678aae6391',
 '6a2482e9af72b55296872b0d',
 '6965bd850ce22095dd7a84ca',
 '69c093d33989aa791c4e8b2d',
 '69a6eb2bc679065ed1cecc51',
 '6978d830c5d415696508430d',
 '6a14434573bcee3f17b7f762',
 '69cc1fac2b52b1c4dd21cc7d']

In [31]:
answers_df = answers_df[~answers_df["participantId"].isin(all_filtered_participants)]

# MERGE DFs TOGETHER

## all the word information back into the response dataframe

In [32]:
# get the weak word info
final_df = answers_df.merge(word_freq_dict_df, left_on="weak", right_on="Word")
final_df = final_df.drop(["Word"], axis=1)
final_df = final_df.rename(columns={
    "Rank": "weakRank",
    "Frequency": "weakFreq",
})

# get the strong word info
final_df = final_df.merge(word_freq_dict_df, left_on="strong", right_on="Word")
final_df = final_df.drop(["Word"], axis=1)
final_df = final_df.rename(columns={
    "Rank": "strongRank",
    "Frequency": "strongFreq",
})

# get the antonym word info
final_df = final_df.merge(word_freq_dict_df, left_on="antonym", right_on="Word")
final_df = final_df.drop(["Word"], axis=1)
final_df = final_df.rename(columns={
    "Rank": "antonymRank",
    "Frequency": "antonymFreq",
})

# create freq ratios
cols = ["weakFreq", "strongFreq", "antonymFreq", "weakRank", "strongRank", "antonymRank"]
final_df[cols] = final_df[cols].apply(pd.to_numeric, errors='coerce', axis=1)
final_df["freqRatio"] = final_df["weakFreq"] / final_df["strongFreq"]
# if freq ratio is nan, fill it with 2
final_df["freqRatio"] = final_df["freqRatio"].apply(lambda x: math.log10(x)).fillna(2, downcast='infer')
final_df.head(6)

/tmp/ipykernel_724073/2728389408.py:30: FutureWarning: The 'downcast' keyword in fillna is deprecated and will be removed in a future version. Use res.infer_objects(copy=False) to infer non-object dtype, or pd.to_numeric with the 'downcast' keyword to downcast numeric results.
  final_df["freqRatio"] = final_df["freqRatio"].apply(lambda x: math.log10(x)).fillna(2, downcast='infer')


,participantId,itemId,itemType,weak,strong,antonym,lowfreq1,lowfreq2,derivationStrength,strongRating,antonymRating,speakerType,speakerGender,weakRank,weakFreq,strongRank,strongFreq,antonymRank,antonymFreq,freqRatio
0,62c5f85ccf34c8d9c0650f6e,1,critical,soft,mushy,crunchy,panicked,introspective,90.0,10.0,0.0,nonnative,male,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
1,661c09a749790db73f8faba5,1,critical,soft,mushy,crunchy,panicked,introspective,35.0,60.0,5.0,native,female,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
2,66315563a848526fcfa7555b,1,critical,soft,mushy,crunchy,panicked,introspective,94.0,6.0,0.0,nonnative,male,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
3,66577b950cf252021eff76f8,1,critical,soft,mushy,crunchy,panicked,introspective,100.0,0.0,0.0,nonnative,male,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
4,66da162f5538535e93490441,1,critical,soft,mushy,crunchy,panicked,introspective,100.0,0.0,0.0,native,female,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
5,676956116c1a3416bf7b2afe,1,critical,soft,mushy,crunchy,panicked,introspective,100.0,0.0,0.0,native,female,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265


In [34]:
final_df.shape

(1980, 20)

## read and merge the ToM scores in

In [35]:
tom_scores_df = pd.read_csv("./dataOutput/ToM/nathan_scores.csv")[["workerid", "nathan_score_full"]]
tom_scores_df = tom_scores_df[~tom_scores_df["workerid"].isin(all_filtered_participants)]

final_tom_df = final_df.merge(tom_scores_df, left_on="participantId", right_on="workerid")

final_tom_df.shape

(1920, 22)

# FINAL OUTPUT

In [36]:
final_remove_cols = ["weakRank", "strongRank", "antonymRank", "weakFreq", "strongFreq", "antonymFreq"]
final_df = final_df.drop(final_remove_cols, axis=1)
final_df.to_csv("./dataOutput/formatted_results.csv")

In [15]:
unformatted_df["answer"].head()

1    "{\"prob1\":0,\"prob2\":\"10\",\"prob3\":90,\"...
2    "{\"prob1\":5,\"prob2\":\"60\",\"prob3\":\"35\...
3    "{\"prob1\":\"94\",\"prob2\":0,\"prob3\":6,\"p...
4    "{\"prob1\":0,\"prob2\":0,\"prob3\":\"100\",\"...
5    "{\"prob1\":0,\"prob2\":\"100\",\"prob3\":0,\"...
Name: answer, dtype: object

In [16]:
sample = unformatted_df.iloc[0]
sample

workerid                             62c5f85ccf34c8d9c0650f6e
answer      "{\"prob1\":0,\"prob2\":\"10\",\"prob3\":90,\"...
itemid                                                      1
type                                                 critical
weak                                                     soft
strong                                                  mushy
antonym                                               crunchy
lowfreq1                                             panicked
lowfreq2                                        introspective
Name: 1, dtype: object

In [17]:
sample_answer = json.loads(json.loads(sample["answer"]))
sample_answer

{'prob1': 0,
 'prob2': '10',
 'prob3': 90,
 'probsum': 100,
 'ratings': [{'image': 'crunchy.png', 'rating': 0},
  {'image': 'mushy.png', 'rating': 10},
  {'image': 'soft.png', 'rating': 90}],
 'speaker': 'ariq',
 'timeTaken': 17915}

In [18]:
new_dict = dict()
for item in sample_answer["ratings"]:
    name = item["image"].replace(".png", "")

    # if name == sample["weak"]:
    #     tag = "target"
    # elif name == sample["strong"]:
    #     tag = "strong"
    # else:
    #     tag = "antonym"
    
    new_dict.update(
        {name: item["rating"]}
    )
new_dict

{'crunchy': 0, 'mushy': 10, 'soft': 90}

In [19]:
if sample_answer["speaker"] in ["frank", "bri"]:
    speakerType = "native"
else:
    speakerType = "nonnative"

if sample_answer["speaker"] in ["bri", "sasha"]:
    speakerGender = "female"
else:
    speakerGender = "male"

speakerGender, speakerType, sample["type"]

('male', 'nonnative', 'critical')

In [20]:
word_freq_dict = pd.read_csv("wordFreqDict.csv")
word_freq_dict.head()

FileNotFoundError: [Errno 2] No such file or directory: 'wordFreqDict.csv'